# Pulling and Cleaning CRSP Monthly Returns

## What this notebook does and why

CRSP (Center for Research in Security Prices) is the gold standard for US equity return data. This notebook pulls monthly stock returns from CRSP via WRDS for every permno in the Kelly et al. (2020) universe, applies a sequence of cleaning decisions that are each independently motivated, and produces `crsp_returns.csv` — the return series that gets merged onto the Kelly signals in the next notebook.

The output of this notebook feeds directly into `merging_init_paper_data_and_crsp_returns.ipynb`. Nothing in the backtest pipeline runs without this file.

## Why we do not pull all of CRSP

CRSP contains roughly 100 million rows across its full history. Pulling everything and filtering afterwards would be slow, memory-intensive, and wasteful. Instead, we extract the unique permnos from the Kelly dataset upfront and pass them as a filter directly into the SQL query. This reduces the pull to approximately 4.2 million rows — a 25x reduction.

## The n+1 timing convention — the single most important thing in this notebook

In the Kelly dataset, the `ret` on row `(permno, date=2020-01)` is the return **earned in February 2020** — the next calendar month. The signals observed at the end of January form the portfolio, and that portfolio earns its return over February.

In raw CRSP, `ret` on `date=2020-02` is also the return earned **during** February 2020.

So the alignment we need is: `Kelly row date=2020-01` ↔ `CRSP row date=2020-02`.

We handle this by shifting CRSP returns **back by one month** within each permno after pulling. After the shift, the return sitting on `date=2020-01` in CRSP is the return earned in January — which is what Kelly calls the return for December. The dates then align correctly for a direct merge on `(permno, date)`.

Getting this wrong introduces look-ahead bias. We verify the alignment explicitly at the end of this notebook by computing the correlation between Kelly's own `ret` column and our shifted CRSP `ret` for a sample of permnos. The expected correlation is 1.000.

## Date range

We pull from 1957 to 2026-03. The Kelly dataset ends December 2021 — but to get the return for December 2021 (which in Kelly's convention sits on the December row), we need CRSP's January 2022 return. The extra months through 2026 cover the extension period (2022-2024) and provide a buffer.

## Step 1 — Load the Kelly universe and extract permnos

Before connecting to WRDS, I load the Kelly dataset to extract the exact list of permnos I need. Every SQL query will be filtered to this list. I also save the permno list to `kelly_permnos.csv` as a reference file in case I need to re-run the CRSP pull separately.

In [ ]:
import wrds
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load the Kelly dataset — we only need the permno column here
# but load_low_memory=True avoids dtype inference warnings on the large file
kelly = pd.read_csv('datashare\\datashare.csv', low_memory=True)

print("Kelly dataset shape:", kelly.shape)
print("Date range:", kelly['DATE'].min(), "to", kelly['DATE'].max())
print("Unique permnos:", kelly['permno'].nunique())
print("Columns:", kelly.columns.tolist())

# Extract the unique permnos — this defines our investment universe
# Every CRSP query will be filtered to exactly these stocks
# Pulling the full CRSP universe (~100M rows) and filtering afterwards would be wasteful
permnos = kelly['permno'].unique().tolist()
print(f"\nTotal unique permnos to pull: {len(permnos)}")

# Save the permno list as a reference — useful if the CRSP pull needs to be re-run
pd.Series(permnos).to_csv('kelly_permnos.csv', index=False)

## Step 2 — Connect to WRDS and verify access

Connecting to WRDS requires university credentials. I verify access to the three tables we need before running any large queries: `crsp.msf` (monthly stock file — the main return data), `crsp.msenames` (name history — needed for share code and exchange filters), and `comp.funda` (Compustat annual fundamentals — needed later for the extension period).

The connection line is commented out here — uncomment and run it once per session.

In [ ]:
# Uncomment to connect — requires WRDS credentials
#conn = wrds.Connection(wrds_username='muhab')

# Verify that we have access to the three tables we need before running any large queries
print("Checking access...")
for lib, table in [('crsp', 'msf'),
                   ('crsp', 'msenames'),
                   ('comp', 'funda')]:
    try:
        test = conn.raw_sql(f"SELECT * FROM {lib}.{table} LIMIT 1")
        print(f"  {lib}.{table}: OK")
    except:
        print(f"  {lib}.{table}: NO ACCESS")

## Step 3 — Pull and clean CRSP returns

### The SQL query — what we pull and why

The main table is `crsp.msf` (monthly stock file). We join two additional tables:

- **`crsp.msenames`** — gives us `shrcd` (share code) and `exchcd` (exchange code). We need these to filter to ordinary common shares on major US exchanges. The join uses a date-valid condition (`namedt <= date <= nameendt`) because share codes and exchange listings change over time.

- **`crsp.msedelist`** — gives us delisting returns (`dlret`) and delisting codes (`dlstcd`). Critically, delisting returns live in a **separate table** in CRSP, not in `msf`. Missing this join means missing the return earned in the final month of a stock's life — which is almost always negative and often catastrophic.

We pull in chunks of 500 permnos to avoid query timeouts on the WRDS server.

### The cleaning function — every decision justified

**Delisting return adjustment (Shumway 1997)**  
When a stock is forcibly delisted (delisting codes 500-599 — bankruptcies, exchanges forcing delistings) and the delisting return is missing in CRSP, we assign -30%. This is the standard academic convention following Shumway (1997), who showed that ignoring delistings inflates returns by approximately 1.5% per year. The -30% is a conservative estimate of the average loss investors experience when a stock is forcibly removed.

**The n+1 shift**  
After pulling, we shift returns back by one month within each permno using `groupby('permno')['ret'].shift(-1)`. This aligns our returns with the Kelly convention where the return on row `(permno, date=T)` is the return earned in month T+1. After the shift, the last month of each permno has `ret = NaN` and is dropped.

**Date standardisation**  
CRSP records dates as the last trading day of the month (e.g. 2020-01-31). Kelly uses the first of the month (2020-01-01). We convert to month-start using `.dt.to_period('M').dt.to_timestamp()` so that a simple merge on `(permno, date)` works correctly.

In [ ]:
def pull_crsp_returns(conn, permnos,
                       start='1957-01-01',
                       end='2026-03-31'):

    # Pull in chunks of 500 permnos to avoid WRDS query timeouts
    chunk_size = 500
    chunks     = [permnos[i:i+chunk_size]
                  for i in range(0, len(permnos), chunk_size)]

    all_chunks = []

    for i, chunk in enumerate(chunks):
        print(f"  Pulling chunk {i+1}/{len(chunks)}...")
        permno_str = ','.join(map(str, chunk))

        chunk_df = conn.raw_sql(f"""
            SELECT
                a.permno,
                a.date,
                a.ret,
                a.retx,
                a.prc,
                a.shrout,
                a.vol,
                b.shrcd,
                b.exchcd,
                b.siccd,
                c.dlret,
                c.dlstcd
            FROM crsp.msf AS a
            -- Join name history for share code and exchange filters
            -- The date condition ensures we get the correct share code
            -- as of each observation date, not just the current one
            LEFT JOIN crsp.msenames AS b
                ON a.permno = b.permno
                AND b.namedt <= a.date
                AND a.date   <= b.nameendt
            -- Join delisting table to capture final-month returns
            -- Delisting returns live in a separate CRSP table — missing this
            -- join means missing catastrophic returns on forced delistings
            LEFT JOIN crsp.msedelist AS c
                ON a.permno = c.permno
                AND date_trunc('month', a.date) = date_trunc('month', c.dlstdt)
            WHERE a.date BETWEEN '{start}' AND '{end}'
                AND a.permno IN ({permno_str})
        """, date_cols=['date'])

        all_chunks.append(chunk_df)

    crsp = pd.concat(all_chunks, ignore_index=True)
    print(f"Total rows: {len(crsp):,}")
    print(f"Unique permnos: {crsp['permno'].nunique():,}")

    return crsp


def clean_crsp_returns(crsp):
    """
    Applies all cleaning steps to raw CRSP data and implements
    the n+1 shift to match the Kelly et al. return convention.
    """

    crsp = crsp.copy()
    crsp = crsp.sort_values(['permno', 'date'])

    # Delisting return adjustment following Shumway (1997)
    # Forced delistings (codes 500-599) with no recorded dlret get -30%
    # Ignoring delistings overstates returns by ~1.5%/yr — not an option
    crsp['dlret'] = crsp['dlret'].fillna(0)
    perf_delist   = crsp['dlstcd'].between(500, 599)
    crsp.loc[perf_delist & (crsp['dlret'] == 0), 'dlret'] = -0.30

    # Combine the regular monthly return with the delisting return
    # The compounding formula: (1 + ret) × (1 + dlret) - 1
    # For non-delisting months dlret = 0 so this is a no-op
    crsp['ret_raw'] = crsp['ret'].copy()
    crsp['ret'] = (
        (1 + crsp['ret'].fillna(0)) *
        (1 + crsp['dlret']) - 1
    )
    # Restore genuine NaN for rows where both ret and dlret were missing
    crsp.loc[crsp['ret_raw'].isna() &
             (crsp['dlret'] == 0), 'ret'] = np.nan

    # Market equity: price × shares outstanding, converted to millions
    # Use absolute value of prc — CRSP uses negative prices for bid-ask midpoints
    crsp['me'] = crsp['prc'].abs() * crsp['shrout'] / 1000

    # The n+1 shift: align CRSP returns with Kelly's timing convention
    # Kelly's ret on date T = the return earned DURING month T+1
    # After this shift, CRSP's ret on date T is also the return earned in T+1
    # A direct merge on (permno, date) then works correctly
    crsp['ret'] = crsp.groupby('permno')['ret'].shift(-1)

    # Standardise dates to month-start (YYYY-MM-01) to match Kelly's DATE format
    # CRSP uses last trading day of month; Kelly uses first of month
    crsp['date'] = pd.to_datetime(crsp['date'])
    crsp['date'] = crsp['date'].dt.to_period('M').dt.to_timestamp()

    # Drop the last month for each permno — ret is NaN after the shift
    crsp = crsp.dropna(subset=['ret'])

    crsp = crsp.sort_values(['permno', 'date']).reset_index(drop=True)

    print("After cleaning:")
    print(f"  Rows: {len(crsp):,}")
    print(f"  Return range: {crsp['ret'].min():.4f} to {crsp['ret'].max():.4f}")
    print(f"  Date range: {crsp['date'].min()} to {crsp['date'].max()}")

    return crsp


# Run the pull
crsp_raw = pull_crsp_returns(conn, permnos)

## Step 4 — Apply cleaning and save

Run the cleaning function on the raw pull and save to disk. This file is used by everything downstream — the Kelly merge notebook, the extension period notebook, and the backtest pipeline.

In [ ]:
crsp_clean = clean_crsp_returns(crsp_raw)

# Save the cleaned file — this is the input for the Kelly merge notebook
crsp_clean.to_csv('crsp_returns.csv', index=False)
print("Saved: crsp_returns.csv")

## Step 5 — Inspect the return distribution and identify outliers

Before applying the price filter and winsorisation, I want to understand what extreme returns look like in the cleaned dataset. A return above 200% in a single month is almost certainly a data issue — a stock splitting, a corporate action being recorded incorrectly, or a stale price that suddenly corrects. I look at the price distribution of those extreme return stocks to confirm they are penny stocks, which motivates the $1 price filter.

In [ ]:
# Reload from disk to work from the saved file
crsp_clean = pd.read_csv('crsp_returns.csv', parse_dates=['date'])

print("Date range:", crsp_clean['date'].min(), "to", crsp_clean['date'].max())
print("Rows:", len(crsp_clean))

print("\nReturn percentiles:")
print(crsp_clean['ret'].quantile([0.001, 0.01, 0.99, 0.999, 0.9999]))

# Look at the top 10 returns to understand what we are dealing with
print("\nTop 10 highest returns:")
print(crsp_clean.nlargest(10, 'ret')[['permno','date','ret','prc','shrout']])

## Step 6 — Price filter and winsorisation

### Price filter: exclude stocks below $1

The extreme return stocks are almost exclusively penny stocks — stocks with prices below $1. At these price levels, a move from $0.02 to $0.05 is a 150% return that looks extraordinary but represents $0.03 of actual price movement. These stocks are untradeable at any meaningful scale, have bid-ask spreads of 20-50% of their price, and produce signal noise rather than genuine alpha. We remove them.

Note that CRSP uses negative prices to indicate bid-ask midpoints rather than transaction prices. We take the absolute value before applying the filter.

### Winsorisation: cap at 1st and 99th percentile cross-sectionally

After the price filter, we winsorise returns within each month's cross-section at the 1st and 99th percentile. This caps residual data errors and extreme values without removing any rows. The winsorisation is cross-sectional — the cap adjusts to each month's own volatility regime — rather than applying a fixed global threshold.

This matches Kelly et al.'s exact preprocessing. The pre-winsorisation return is preserved in `ret_raw` for diagnostics.

In [ ]:
# Inspect the price distribution of the most extreme return stocks
# This motivates the $1 price filter below
print("Price distribution of extreme return stocks:")
extreme = crsp_clean[crsp_clean['ret'] > 2.0]
print(f"Rows with ret > 200%: {len(extreme)}")
print(extreme['prc'].abs().describe())

# Price filter: remove stocks trading below $1
# At sub-$1 prices, small absolute moves produce huge percentage returns
# These are untradeable at scale and produce noise not signal
# CRSP uses negative prc for bid-ask midpoints — take absolute value
crsp_clean['prc_abs']  = crsp_clean['prc'].abs()
low_price_mask         = crsp_clean['prc_abs'] < 1.0

print(f"\nRows removed by price filter (prc < $1): {low_price_mask.sum():,}")
print(f"As % of total: {low_price_mask.mean():.2%}")

crsp_filtered = crsp_clean[~low_price_mask].copy()

# Cross-sectional winsorisation at 1st/99th percentile within each month
# This caps residual data errors without removing rows
# Cross-sectional (not global) so the cap adapts to each month's volatility regime
# Matching Kelly et al.'s exact preprocessing
p01 = crsp_filtered.groupby('date')['ret'].transform(
    lambda x: x.quantile(0.01)
)
p99 = crsp_filtered.groupby('date')['ret'].transform(
    lambda x: x.quantile(0.99)
)

# Preserve the pre-winsorisation return for diagnostics
crsp_filtered['ret_raw'] = crsp_filtered['ret'].copy()
crsp_filtered['ret']     = crsp_filtered['ret'].clip(lower=p01, upper=p99)

print("\nAfter filtering:")
print(f"  Rows: {len(crsp_filtered):,}")
print(f"  Removed: {len(crsp_clean) - len(crsp_filtered):,}")
print(f"  Return range: {crsp_filtered['ret'].min():.4f} to {crsp_filtered['ret'].max():.4f}")
print(f"\nReturn percentiles after:")
print(crsp_filtered['ret'].quantile([0.001, 0.01, 0.99, 0.999]))

## Step 7 — Overwrite the saved file with the filtered version

We overwrite `crsp_returns.csv` with the price-filtered and winsorised version. This is the final clean file that the rest of the pipeline uses.

In [ ]:
# Overwrite the saved file with the fully cleaned version
# This is the final crsp_returns.csv that all downstream notebooks read
crsp_filtered.to_csv('crsp_returns.csv', index=False)
print("\nSaved: crsp_returns.csv")

## Step 8 — Verify alignment with the Kelly dataset

This is the most important validation step. The n+1 shift is critical — if it was applied incorrectly, the entire backtest has look-ahead bias baked in at the data level.

I verify by taking 3 random permnos and computing the correlation between Kelly's own `ret` column and our CRSP `ret` on the same `(permno, date)` pairs. Since Kelly's `ret` is the return for the following month, and our CRSP `ret` has been shifted to match that convention, the correlation should be exactly 1.000 for all three.

A correlation below 0.999 would indicate the shift was applied incorrectly and the entire pipeline needs to be rerun.

In [ ]:
# Convert Kelly dates to match CRSP format for the comparison
kelly['date'] = pd.to_datetime(kelly['DATE'].astype(str), format='%Y%m%d')
kelly['date'] = kelly['date'].dt.to_period('M').dt.to_timestamp()

# Sample 3 random permnos and compare Kelly ret vs our CRSP ret on matching dates
# Expected correlation: 1.000 — if it is lower, the n+1 shift is wrong
test_permnos = kelly['permno'].sample(3, random_state=42).values
print("\nVerifying n+1 alignment with Kelly dataset...")
print("Expected: correlation = 1.000 for all permnos\n")

for p in test_permnos:
    k = (kelly[kelly['permno'] == p][['date','ret']]
         .sort_values('date')
         .head(8)
         .rename(columns={'ret': 'kelly_ret'}))

    c = (crsp_filtered[crsp_filtered['permno'] == p][['date','ret']]
         .sort_values('date')
         .head(8)
         .rename(columns={'ret': 'crsp_ret'}))

    comp = k.merge(c, on='date', how='inner')

    if len(comp) > 0:
        corr = comp['kelly_ret'].corr(comp['crsp_ret'])
        print(f"permno {p}: {len(comp)} matched rows, correlation = {corr:.6f}")
        print(comp.head(5).to_string(index=False))
        print()